In [1]:
library(tidyverse)
library(xgboost)

base <- "C:/Users/tyler/OneDrive/Documents/GitHub/work_to_show/Classes Spring 2026/Econometrics/submissions/Walmart Recruiting - Store Sales Forecasting/data/"

train    <- read.csv(paste0(base, "train.csv"))
test     <- read.csv(paste0(base, "test.csv"))
features <- read.csv(paste0(base, "features.csv"))
stores   <- read.csv(paste0(base, "stores.csv"))

Warning message:
"package 'tidyverse' was built under R version 4.5.2"
Warning message:
"package 'ggplot2' was built under R version 4.5.2"
Warning message:
"package 'tidyr' was built under R version 4.5.2"
Warning message:
"package 'dplyr' was built under R version 4.5.2"
Warning message:
"package 'stringr' was built under R version 4.5.2"
── Attaching core tidyverse packages ────────────────────────────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.2     ✔ tibble    3.3.0
✔ lubridate 1.9.4     ✔ tidyr     1.3.2
✔ purrr     1.1.0     
── Conflicts ──────────────────────────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
Warning message:
"package 'xgboost' was built under R version 4.5.3"


In [11]:
# Parse dates
train$Date    <- as.Date(train$Date)
test$Date     <- as.Date(test$Date)
features$Date <- as.Date(features$Date)

# Drop IsHoliday from features before merging to avoid duplicate columns
features <- features %>% select(-IsHoliday)

# Merge
train <- train %>%
  left_join(features, by = c("Store", "Date")) %>%
  left_join(stores,   by = "Store")

test <- test %>%
  left_join(features, by = c("Store", "Date")) %>%
  left_join(stores,   by = "Store")

In [12]:
# Holiday flag & WMAE weights (use the IsHoliday already in train/test)
train$IsHoliday <- as.integer(train$IsHoliday)
test$IsHoliday  <- as.integer(test$IsHoliday)
train$weight    <- ifelse(train$IsHoliday == 1, 5, 1)

In [13]:
# Feature engineering
engineer <- function(df) {
  df$week    <- as.integer(format(df$Date, "%U"))
  df$month   <- as.integer(format(df$Date, "%m"))
  df$year    <- as.integer(format(df$Date, "%Y"))
  df$quarter <- as.integer(substr(quarters(df$Date), 2, 2))
  df$Type    <- as.integer(factor(df$Type, levels = c("A","B","C")))

  md_cols <- paste0("MarkDown", 1:5)
  for (col in md_cols) {
    if (col %in% colnames(df)) df[[col]][is.na(df[[col]])] <- 0
  }

  for (col in colnames(df)) {
    if (is.numeric(df[[col]]) && any(is.na(df[[col]]))) {
      df[[col]][is.na(df[[col]])] <- median(df[[col]], na.rm = TRUE)
    }
  }
  df
}

In [14]:
train <- engineer(train)
test  <- engineer(test)

# Sanity check
cat("Train NAs:", sum(is.na(train)), "\n")
cat("Test NAs:",  sum(is.na(test)),  "\n")
cat("Train rows:", nrow(train), "\n")
cat("Test rows:",  nrow(test),  "\n")

feat_cols <- c("Store", "Dept", "week", "month", "year", "quarter",
               "IsHoliday", "Type", "Size",
               "Temperature", "Fuel_Price", "CPI", "Unemployment",
               "MarkDown1", "MarkDown2", "MarkDown3", "MarkDown4", "MarkDown5")

X_train <- as.matrix(train[, feat_cols])
y_train <- train$Weekly_Sales
w_train <- train$weight
X_test  <- as.matrix(test[, feat_cols])

Train NAs: 0 
Test NAs: 0 
Train rows: 421570 
Test rows: 115064 


In [15]:
dtrain <- xgb.DMatrix(X_train, label = y_train, weight = w_train)
dtest  <- xgb.DMatrix(X_test)

params <- list(
  objective        = "reg:squarederror",
  eta              = 0.05,
  max_depth        = 6,
  subsample        = 0.8,
  colsample_bytree = 0.8,
  min_child_weight = 10
)

set.seed(42)
model <- xgb.train(
  params  = params,
  data    = dtrain,
  nrounds = 500,
  verbose = 1
)

In [16]:
preds <- data.frame(
  Id           = paste(test$Store, test$Dept, format(test$Date, "%Y-%m-%d"), sep = "_"),
  Weekly_Sales = predict(model, dtest)
)

In [17]:
out <- "C:/Users/tyler/OneDrive/Documents/GitHub/work_to_show/Classes Spring 2026/Econometrics/submissions/Walmart Recruiting - Store Sales Forecasting/"
write.csv(preds, paste0(out, "submission_xgb.csv"), row.names = FALSE)

In [10]:
base <- "C:/Users/tyler/OneDrive/Documents/GitHub/work_to_show/Classes Spring 2026/Econometrics/submissions/Walmart Recruiting - Store Sales Forecasting/data/"

train    <- read.csv(paste0(base, "train.csv"))
test     <- read.csv(paste0(base, "test.csv"))
features <- read.csv(paste0(base, "features.csv"))
stores   <- read.csv(paste0(base, "stores.csv"))

cat("=== TRAIN ===\n")
print(head(train, 3))
print(dim(train))

cat("=== TEST ===\n")
print(head(test, 3))
print(dim(test))

cat("=== FEATURES ===\n")
print(head(features, 3))
print(colnames(features))

cat("=== STORES ===\n")
print(head(stores, 3))

=== TRAIN ===
  Store Dept       Date Weekly_Sales IsHoliday
1     1    1 2010-02-05     24924.50     FALSE
2     1    1 2010-02-12     46039.49      TRUE
3     1    1 2010-02-19     41595.55     FALSE
[1] 421570      5
=== TEST ===
  Store Dept       Date IsHoliday
1     1    1 2012-11-02     FALSE
2     1    1 2012-11-09     FALSE
3     1    1 2012-11-16     FALSE
[1] 115064      4
=== FEATURES ===
  Store       Date Temperature Fuel_Price MarkDown1 MarkDown2 MarkDown3
1     1 2010-02-05       42.31      2.572        NA        NA        NA
2     1 2010-02-12       38.51      2.548        NA        NA        NA
3     1 2010-02-19       39.93      2.514        NA        NA        NA
  MarkDown4 MarkDown5      CPI Unemployment IsHoliday
1        NA        NA 211.0964        8.106     FALSE
2        NA        NA 211.2422        8.106      TRUE
3        NA        NA 211.2891        8.106     FALSE
 [1] "Store"        "Date"         "Temperature"  "Fuel_Price"   "MarkDown1"   
 [6] "MarkDo

In [23]:
library(dplyr)
library(purrr)
library(lubridate)
library(slider)
library(tidyr)

# -----------------------------
# 1. CREATE SAFE ID
# -----------------------------
train$ID <- paste(train$Store, train$Dept, sep = "_")
test$ID  <- paste(test$Store, test$Dept, sep = "_")

# -----------------------------
# 2. FEATURE ENGINEERING (SAFE FOR TEST)
# -----------------------------
make_features <- function(df, is_train = TRUE) {

  df %>%
    arrange(ID, Date) %>%
    group_by(ID) %>%
    mutate(
      week  = lubridate::week(Date),
      month = lubridate::month(Date),
      year  = lubridate::year(Date),
      Holiday = as.numeric(IsHoliday),

      # SAFE LAGS (only exist in train)
      lag_1 = if (is_train) lag(Weekly_Sales, 1) else 0,
      lag_2 = if (is_train) lag(Weekly_Sales, 2) else 0,
      lag_4 = if (is_train) lag(Weekly_Sales, 4) else 0,

      roll_4 = if (is_train)
        slider::slide_dbl(Weekly_Sales, mean, .before = 3, .complete = FALSE)
      else 0
    ) %>%
    ungroup()
}

train <- make_features(train, TRUE)
test  <- make_features(test, FALSE)

# -----------------------------
# 3. CLEAN DATA (IMPORTANT)
# -----------------------------
train <- train %>% mutate(across(where(is.numeric), ~replace_na(., 0)))
test  <- test  %>% mutate(across(where(is.numeric), ~replace_na(., 0)))

# -----------------------------
# 4. MODELING
# -----------------------------
ids <- unique(test$ID)

preds <- map_dfr(ids, function(id) {

  tr <- train[train$ID == id, ]
  te <- test[test$ID == id, ]

  # fallback if no history
  if (nrow(tr) < 5) {
    return(data.frame(
      Id = paste(te$Store, te$Dept, format(te$Date, "%Y-%m-%d"), sep = "_"),
      Weekly_Sales = 0
    ))
  }

  predictors <- c(
    "week", "month", "year",
    "Holiday",
    "lag_1", "lag_2", "lag_4",
    "roll_4"
  )

  fmla <- as.formula(
    paste("Weekly_Sales ~", paste(predictors, collapse = " + "))
  )

  fit <- tryCatch(
    lm(fmla, data = tr),
    error = function(e) NULL
  )

  yhat <- if (!is.null(fit)) {
    predict(fit, newdata = te)
  } else {
    rep(mean(tr$Weekly_Sales, na.rm = TRUE), nrow(te))
  }

  # FINAL SAFETY CLEANUP
  yhat <- as.numeric(yhat)
  yhat[is.na(yhat)] <- mean(tr$Weekly_Sales, na.rm = TRUE)
  yhat[yhat < 0] <- 0

  data.frame(
    Id = paste(te$Store, te$Dept, format(te$Date, "%Y-%m-%d"), sep = "_"),
    Weekly_Sales = yhat
  )
})

# -----------------------------
# 5. FINAL SUBMISSION CLEAN
# -----------------------------
preds <- preds %>%
  filter(!is.na(Id)) %>%
  distinct(Id, .keep_all = TRUE)

preds$Weekly_Sales <- as.numeric(preds$Weekly_Sales)
preds$Weekly_Sales[is.na(preds$Weekly_Sales)] <- 0

# -----------------------------
# 6. SAVE
# -----------------------------
out <- "C:/Users/tyler/OneDrive/Documents/GitHub/work_to_show/Classes Spring 2026/Econometrics/submissions/Walmart Recruiting - Store Sales Forecasting/"

write.csv(preds, paste0(out, "submission_10.csv"), row.names = FALSE)

cat("✅ CLEAN SUBMISSION READY\n")

Warning message in predict.lm(fit, newdata = te):
"prediction from rank-deficient fit; attr(*, "non-estim") has doubtful cases"
Warning message in predict.lm(fit, newdata = te):
"prediction from rank-deficient fit; attr(*, "non-estim") has doubtful cases"
Warning message in predict.lm(fit, newdata = te):
"prediction from rank-deficient fit; attr(*, "non-estim") has doubtful cases"
Warning message in predict.lm(fit, newdata = te):
"prediction from rank-deficient fit; attr(*, "non-estim") has doubtful cases"
Warning message in predict.lm(fit, newdata = te):
"prediction from rank-deficient fit; attr(*, "non-estim") has doubtful cases"
Warning message in predict.lm(fit, newdata = te):
"prediction from rank-deficient fit; attr(*, "non-estim") has doubtful cases"
Warning message in predict.lm(fit, newdata = te):
"prediction from rank-deficient fit; attr(*, "non-estim") has doubtful cases"
Warning message in predict.lm(fit, newdata = te):
"prediction from rank-deficient fit; attr(*, "non-esti

✅ CLEAN SUBMISSION READY
